## LSTM for Text Generation

Long Short-Term Memory (LSTM) is a specialized recurrent neural network (RNN) designed to learn long-term dependencies by resolving the vanishing gradient problem.

### Application Areas
* Language modeling
* Text analysis & auto-completion
* Time-series & stock price forecasting

### Problem Definition: Next-Word Prediction
We will train a character/word-level LSTM to predict the next word given a starting word, using a sample product review as our dataset.

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter # For calculating word frequencies
from itertools import product # For generating combinations in grid search

In [48]:
# 1. Data Loading and Preprocessing

# Product reviews
text = """This product exceeded my expectations. The material quality is very sturdy and it is extremely practical to use. My package arrived just one day after I placed the order and
The performance it offers for its price is truly surprisingly good. I wholeheartedly recommend this product to everyone around me!"""

# Data preprocessing:
"""
remove punctuation,
convert to lower case,
split words
"""

words = text.replace(".","").replace("!","").lower().split()

# Calculate word frequencies and build indexing
word_counts = Counter(words)
vocab = sorted(word_counts, key=word_counts.get, reverse=True) # Sort word frequencies from highest to lowest
word2ix = {word: i for i, word in enumerate(vocab)}
ix2word = {i: word for i, word in enumerate(vocab)}

# Prepare training data
data = [(words[i], words[i+1]) for i in range(len(words) - 1)]

In [49]:
# 2. LSTM Model Definition
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTM, self).__init__() # Call the superclass constructor
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # Embedding layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim) # LSTM layer
        self.fc = nn.Linear(hidden_dim, vocab_size) # Fully connected layer (output)

    def forward(self, x):
        """
        input -> embedding -> lstm -> fc -> output
        """
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x.view(1, 1, -1))
        output = self.fc(lstm_out.view(1, -1))
        return output

In [50]:
model = LSTM(len(vocab), embedding_dim=8, hidden_dim=32)

In [51]:
# 3. Hyperparameter Tuning (Grid Search)

# Word list -> tensor
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

# Determine hyperparameter tuning combinations
embedding_sizes = [8, 16] # Embedding sizes to try
hidden_sizes = [32, 64] # Hidden layer sizes to try
learning_rates = [0.01, 0.005] # Learning rates

best_loss = float('inf') # Variable to store the lowest loss value
best_params = {} # Empty dictionary to store the best parameters

print("Hyperparameter Tuning is starting . . .")

# Grid search
for embedding_size, hidden_size, learning_rate in product(embedding_sizes, hidden_sizes, learning_rates):
    print(f"Embedding Size: {embedding_size}, Hidden Size: {hidden_size}, Learning Rate: {learning_rate}")

    # Define model
    model = LSTM(len(vocab), embedding_dim=embedding_size, hidden_dim=hidden_size) # Create model with selected parameters
    loss_function = nn.CrossEntropyLoss() # Loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate) # Optimizer

    epochs = 50
    total_loss = 0
    for epoch in range(epochs):
        epoch_loss = 0
        for word, next_word in data:
            model.zero_grad() # Reset gradients
            input_tensor = prepare_sequence([word], word2ix) # Convert input to tensor
            target_tensor = prepare_sequence([next_word], word2ix) # Convert target to tensor
            output = model(input_tensor) # Prediction
            loss = loss_function(output, target_tensor) # Calculate loss
            loss.backward() # Calculate gradients
            optimizer.step() # Update parameters
            epoch_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch: {epoch}, Loss: {epoch_loss:.5f}")
        total_loss += epoch_loss

    if total_loss < best_loss:
        best_loss = total_loss
        best_params = {
            'embedding_dim': embedding_size,
            'hidden_dim': hidden_size,
            'learning_rate': learning_rate
        }
    print()

print(f"Best params: {best_params}")

Hyperparameter Tuning is starting . . .
Embedding Size: 8, Hidden Size: 32, Learning Rate: 0.01
Epoch: 0, Loss: 184.05066
Epoch: 10, Loss: 27.29880
Epoch: 20, Loss: 22.99529
Epoch: 30, Loss: 21.73109
Epoch: 40, Loss: 20.86818

Embedding Size: 8, Hidden Size: 32, Learning Rate: 0.005
Epoch: 0, Loss: 181.76589
Epoch: 10, Loss: 36.44139
Epoch: 20, Loss: 22.87031
Epoch: 30, Loss: 20.79829
Epoch: 40, Loss: 19.95562

Embedding Size: 8, Hidden Size: 64, Learning Rate: 0.01
Epoch: 0, Loss: 182.45947
Epoch: 10, Loss: 27.22732
Epoch: 20, Loss: 22.83790
Epoch: 30, Loss: 21.43844
Epoch: 40, Loss: 20.79418

Embedding Size: 8, Hidden Size: 64, Learning Rate: 0.005
Epoch: 0, Loss: 181.46957
Epoch: 10, Loss: 26.81695
Epoch: 20, Loss: 22.46641
Epoch: 30, Loss: 21.11277
Epoch: 40, Loss: 20.36384

Embedding Size: 16, Hidden Size: 32, Learning Rate: 0.01
Epoch: 0, Loss: 183.05663
Epoch: 10, Loss: 25.72964
Epoch: 20, Loss: 22.81889
Epoch: 30, Loss: 21.93931
Epoch: 40, Loss: 20.99612

Embedding Size: 16, Hi

In [52]:
# 4. Final Model Training

final_model = LSTM(len(vocab), best_params['embedding_dim'], best_params['hidden_dim'])
optimizer = optim.Adam(final_model.parameters(), lr=best_params['learning_rate'])
loss_function = nn.CrossEntropyLoss()

print("Training final model...")
epochs = 100
for epoch in range(epochs):
    epoch_loss = 0
    for word, next_word in data:
        final_model.zero_grad()
        input_tensor = prepare_sequence([word], word2ix)
        target_tensor = prepare_sequence([next_word], word2ix)
        output = final_model(input_tensor)
        loss = loss_function(output, target_tensor)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:02d} | Loss: {epoch_loss:.5f}")

Training final model...
Epoch: 00 | Loss: 183.12777
Epoch: 10 | Loss: 25.54541
Epoch: 20 | Loss: 22.27997
Epoch: 30 | Loss: 20.90573
Epoch: 40 | Loss: 20.48761
Epoch: 50 | Loss: 19.54045
Epoch: 60 | Loss: 19.31145
Epoch: 70 | Loss: 18.84368
Epoch: 80 | Loss: 18.55662
Epoch: 90 | Loss: 17.89440


In [53]:
# 5. Text Prediction and Generation Function

# Word prediction function: takes a starting word and generates n words
def predict_sequence(start_word, num_words):
    current_word = start_word # Current word is set as the starting word
    output_sequence = [current_word] # Output sequence

    for _ in range(num_words): # Predict specified number of words
        with torch.no_grad():
            input_tensor = prepare_sequence([current_word], word2ix) # Word -> tensor
            output = final_model(input_tensor) # Prediction
            predicted_idx = torch.argmax(output).item() # Get index of highest predicted word
            predicted_word = ix2word[predicted_idx] # Index -> word
            output_sequence.append(predicted_word) # Add prediction to output sequence
            current_word = predicted_word # Update current word for the next prediction

    return output_sequence # Return predicted word sequence

In [54]:
# 6. Run Prediction Demo

start_word = "this"
num_predictions = 5
predicted_sequence = predict_sequence(start_word, num_predictions)

print(f"Start Word: {start_word}")
print(f"Generated Sequence: {' '.join(predicted_sequence)}")

Start Word: this
Generated Sequence: this product to use my package
